# 🏭 Project 9.1 — Factory Floor Network (BFS & DFS)
**DSA for Mechatronics · Week 09 — Graphs**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [1]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "230615005"   # ← replace with your real student ID

import random, hashlib, heapq
from collections import defaultdict, deque, Counter
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


Student ID  : 230615005
Dataset seed: 1244338015
✅ Your personal dataset is ready.


## The Scenario
A smart factory has workstations connected by conveyor paths.
We model the layout as an **undirected graph**:
- **Vertices** = workstations
- **Edges** = conveyor paths between them

We need to answer:
1. Is there a path between two stations? (reachability)
2. What is the **shortest path** (fewest hops) between two stations? (BFS)
3. What order would a maintenance robot visit all reachable stations? (DFS)
4. How many **connected components** exist (isolated islands)?


## Step 1 — Build the factory graph (adjacency list + matrix)

In [2]:
# ── Personalised parameters ──────────────────────────────────────
N_STATIONS = random.randint(10, 16)
EDGE_PROB   = round(random.uniform(0.25, 0.45), 2)

STATION_NAMES = [f"ST{i+1:02d}" for i in range(N_STATIONS)]

# Build random connected graph
edges = set()
# first ensure connectivity: random spanning tree
order = list(range(N_STATIONS))
random.shuffle(order)
for i in range(1, N_STATIONS):
    u, v = order[i-1], order[i]
    if u != v:
        edges.add((min(u,v), max(u,v)))
# add extra edges
for u in range(N_STATIONS):
    for v in range(u+1, N_STATIONS):
        if (u, v) not in edges and random.random() < EDGE_PROB:
            edges.add((u, v))

edges = list(edges)

# Adjacency list
adj = defaultdict(list)
for u, v in edges:
    adj[u].append(v)
    adj[v].append(u)
for node in adj:
    adj[node].sort()

# Adjacency matrix
matrix = [[0]*N_STATIONS for _ in range(N_STATIONS)]
for u, v in edges:
    matrix[u][v] = 1
    matrix[v][u] = 1

print(f"Factory floor network:")
print(f"  Stations  : {N_STATIONS}")
print(f"  Paths     : {len(edges)}")
print(f"  Density   : {len(edges) / (N_STATIONS*(N_STATIONS-1)//2):.2f}")
print()
print(f"  Adjacency list:")
for i in range(N_STATIONS):
    nbrs = [STATION_NAMES[j] for j in adj[i]]
    print(f"    {STATION_NAMES[i]}: {nbrs}")
print()
print(f"  Adjacency matrix (1=path exists):")
header = "       " + "  ".join(f"{STATION_NAMES[j]}" for j in range(N_STATIONS))
print(f"  {header}")
for i in range(N_STATIONS):
    row = "  ".join(str(matrix[i][j]) for j in range(N_STATIONS))
    print(f"  {STATION_NAMES[i]}:  {row}")


Factory floor network:
  Stations  : 16
  Paths     : 54
  Density   : 0.45

  Adjacency list:
    ST01: ['ST03', 'ST05', 'ST06', 'ST10', 'ST11', 'ST14']
    ST02: ['ST03', 'ST04', 'ST05', 'ST06', 'ST11', 'ST13', 'ST14']
    ST03: ['ST01', 'ST02', 'ST07', 'ST10', 'ST11', 'ST15', 'ST16']
    ST04: ['ST02', 'ST06', 'ST10', 'ST11', 'ST12', 'ST14', 'ST15', 'ST16']
    ST05: ['ST01', 'ST02', 'ST11', 'ST12', 'ST13']
    ST06: ['ST01', 'ST02', 'ST04', 'ST07', 'ST08', 'ST12', 'ST13', 'ST15', 'ST16']
    ST07: ['ST03', 'ST06', 'ST08', 'ST09', 'ST11', 'ST12', 'ST16']
    ST08: ['ST06', 'ST07', 'ST09', 'ST14']
    ST09: ['ST07', 'ST08', 'ST13']
    ST10: ['ST01', 'ST03', 'ST04', 'ST11', 'ST12', 'ST14', 'ST16']
    ST11: ['ST01', 'ST02', 'ST03', 'ST04', 'ST05', 'ST07', 'ST10', 'ST13']
    ST12: ['ST04', 'ST05', 'ST06', 'ST07', 'ST10', 'ST13', 'ST14']
    ST13: ['ST02', 'ST05', 'ST06', 'ST09', 'ST11', 'ST12', 'ST15', 'ST16']
    ST14: ['ST01', 'ST02', 'ST04', 'ST08', 'ST10', 'ST12', 'ST15', 'ST16']

## Step 2 — BFS: shortest path (fewest hops)

In [3]:
def bfs_shortest_path(adj, src, dst, n):
    """
    BFS from src to find shortest (fewest-hop) path to dst.

    Algorithm:
      1. Queue starts with [src]. visited set to track seen nodes.
      2. Dequeue node u; for each neighbour v not yet visited:
         a. Mark v visited, record prev[v] = u, enqueue v.
         b. If v == dst, stop — found shortest path.
      3. Reconstruct path by following prev[] from dst back to src.

    Returns (distance, path_list) or (inf, []) if unreachable.
    O(V + E) time.
    """
    if src == dst:
        return 0, [src]
    visited = {src}
    prev    = {src: None}
    queue   = deque([src])
    while queue:
        u = queue.popleft()
        for v in adj[u]:
            if v not in visited:
                visited.add(v)
                prev[v] = u
                if v == dst:
                    # reconstruct
                    path, cur = [], dst
                    while cur is not None:
                        path.append(cur)
                        cur = prev[cur]
                    return len(path)-1, path[::-1]
                queue.append(v)
    return float('inf'), []

# Pick a source and several destinations
src_node = 0
targets  = random.sample(list(range(1, N_STATIONS)), min(5, N_STATIONS-1))

print(f"BFS shortest paths from {STATION_NAMES[src_node]}:")
print(f"  {'Destination':<10}  {'Hops':>5}  Path")
print(f"  {'─'*10}  {'─'*5}  {'─'*40}")
bfs_results = {}
for dst in targets:
    dist, path = bfs_shortest_path(adj, src_node, dst, N_STATIONS)
    path_str   = " → ".join(STATION_NAMES[p] for p in path) if path else "unreachable"
    bfs_results[dst] = (dist, path)
    dist_str   = str(dist) if dist != float('inf') else "∞"
    print(f"  {STATION_NAMES[dst]:<10}  {dist_str:>5}  {path_str}")

avg_dist = round(sum(d for d,_ in bfs_results.values() if d != float('inf')) /
                 max(1, sum(1 for d,_ in bfs_results.values() if d != float('inf'))), 2)
print(f"\n  Average hop distance: {avg_dist}")


BFS shortest paths from ST01:
  Destination   Hops  Path
  ──────────  ─────  ────────────────────────────────────────
  ST08            2  ST01 → ST06 → ST08
  ST03            1  ST01 → ST03
  ST02            2  ST01 → ST03 → ST02
  ST05            1  ST01 → ST05
  ST09            3  ST01 → ST03 → ST07 → ST09

  Average hop distance: 1.8


## Step 3 — DFS: full traversal and cycle detection

In [4]:
def dfs_iterative(adj, start, n):
    """
    Iterative DFS using an explicit stack.
    Returns the order in which nodes are first visited.
    """
    visited = set()
    stack   = [start]
    order   = []
    while stack:
        node = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        order.append(node)
        # push neighbours in reverse so smaller indices are visited first
        for nbr in reversed(adj[node]):
            if nbr not in visited:
                stack.append(nbr)
    return order

def has_cycle_undirected(adj, n):
    """
    Detect cycle in undirected graph via DFS.
    Key: track the parent node; an edge back to a non-parent visited node = cycle.
    """
    visited = set()
    def dfs(node, parent):
        visited.add(node)
        for nbr in adj[node]:
            if nbr not in visited:
                if dfs(nbr, node): return True
            elif nbr != parent:
                return True   # back edge to non-parent = cycle
        return False
    for node in range(n):
        if node not in visited:
            if dfs(node, -1): return True
    return False

dfs_order = dfs_iterative(adj, src_node, N_STATIONS)
has_cycle  = has_cycle_undirected(adj, N_STATIONS)

print(f"DFS traversal from {STATION_NAMES[src_node]}:")
print(f"  Visit order: {' → '.join(STATION_NAMES[i] for i in dfs_order)}")
print(f"  Nodes reached: {len(dfs_order)} / {N_STATIONS}")
print()
print(f"Cycle detection:")
print(f"  Graph has a cycle: {'YES ✅' if has_cycle else 'NO (spanning tree only)'}")
if has_cycle:
    print(f"  (The network has at least one alternative conveyor route.)")
else:
    print(f"  (Every station is reached by exactly one path — tree structure.)")


DFS traversal from ST01:
  Visit order: ST01 → ST03 → ST02 → ST04 → ST06 → ST07 → ST08 → ST09 → ST13 → ST05 → ST11 → ST10 → ST12 → ST14 → ST15 → ST16
  Nodes reached: 16 / 16

Cycle detection:
  Graph has a cycle: YES ✅
  (The network has at least one alternative conveyor route.)


## Step 4 — Connected components

In [5]:
def connected_components(adj, n):
    """
    Find all connected components using BFS.
    Returns list of components, each a sorted list of node indices.
    """
    visited    = set()
    components = []
    for start in range(n):
        if start not in visited:
            component = []
            queue     = deque([start])
            visited.add(start)
            while queue:
                node = queue.popleft()
                component.append(node)
                for nbr in adj[node]:
                    if nbr not in visited:
                        visited.add(nbr)
                        queue.append(nbr)
            components.append(sorted(component))
    return components

components = connected_components(adj, N_STATIONS)

print(f"Connected components:")
print(f"  Total components: {len(components)}")
print()
for i, comp in enumerate(components, 1):
    names = [STATION_NAMES[j] for j in comp]
    print(f"  Component {i}: {names}  ({len(comp)} stations)")

largest  = max(components, key=len)
isolated = [c for c in components if len(c) == 1]
print(f"\n  Largest component : {[STATION_NAMES[j] for j in largest]}")
print(f"  Isolated stations : {len(isolated)}")
print()

# BFS level-by-level distance map from station 0
dist_map = {}
queue = deque([(src_node, 0)])
visited_d = {src_node}
while queue:
    node, d = queue.popleft()
    dist_map[node] = d
    for nbr in adj[node]:
        if nbr not in visited_d:
            visited_d.add(nbr)
            queue.append((nbr, d+1))

print(f"BFS distance from {STATION_NAMES[src_node]} to all reachable stations:")
print(f"  {'Station':<10}  {'Hops':>6}")
print(f"  {'─'*10}  {'─'*6}")
for node in sorted(dist_map):
    print(f"  {STATION_NAMES[node]:<10}  {dist_map[node]:6d}")
max_hop_node = max(dist_map, key=dist_map.get)
print(f"\n  Farthest station  : {STATION_NAMES[max_hop_node]}  ({dist_map[max_hop_node]} hops)")


Connected components:
  Total components: 1

  Component 1: ['ST01', 'ST02', 'ST03', 'ST04', 'ST05', 'ST06', 'ST07', 'ST08', 'ST09', 'ST10', 'ST11', 'ST12', 'ST13', 'ST14', 'ST15', 'ST16']  (16 stations)

  Largest component : ['ST01', 'ST02', 'ST03', 'ST04', 'ST05', 'ST06', 'ST07', 'ST08', 'ST09', 'ST10', 'ST11', 'ST12', 'ST13', 'ST14', 'ST15', 'ST16']
  Isolated stations : 0

BFS distance from ST01 to all reachable stations:
  Station       Hops
  ──────────  ──────
  ST01             0
  ST02             2
  ST03             1
  ST04             2
  ST05             1
  ST06             1
  ST07             2
  ST08             2
  ST09             3
  ST10             1
  ST11             1
  ST12             2
  ST13             2
  ST14             1
  ST15             2
  ST16             2

  Farthest station  : ST09  (3 hops)


## 📋 Final Report

In [6]:
W = 56
print("╔" + "═"*W + "╗")
print("║" + " FACTORY FLOOR NETWORK — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<26}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<26}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  GRAPH STRUCTURE" + " "*(W-17) + "║")
print(f"║  {'Stations (vertices)':<26}: {N_STATIONS:<26}║")
print(f"║  {'Conveyor paths (edges)':<26}: {len(edges):<26}║")
density = round(len(edges) / (N_STATIONS*(N_STATIONS-1)//2), 2)
print(f"║  {'Graph density':<26}: {density:<26}║")
print(f"║  {'Connected components':<26}: {len(components):<26}║")
print(f"║  {'Has cycle':<26}: {'YES' if has_cycle else 'NO':<26}║")
print("╠" + "═"*W + "╣")
print("║  BFS RESULTS" + " "*(W-13) + "║")
print(f"║  {'Source station':<26}: {STATION_NAMES[src_node]:<26}║")
print(f"║  {'Targets checked':<26}: {len(targets):<26}║")
print(f"║  {'Avg hop distance':<26}: {avg_dist:<26}║")
print(f"║  {'Farthest reachable':<26}: {STATION_NAMES[max_hop_node]} ({dist_map[max_hop_node]} hops){'':<10}║")
print("╠" + "═"*W + "╣")
print("║  DFS RESULTS" + " "*(W-13) + "║")
print(f"║  {'Stations visited':<26}: {len(dfs_order)} / {N_STATIONS}{'':<18}║")
visit_str = "→".join(STATION_NAMES[i] for i in dfs_order[:6])
print(f"║  {'Visit order (first 6)':<26}: {visit_str[:26]:<26}║")
print("╚" + "═"*W + "╝")


╔════════════════════════════════════════════════════════╗
║             FACTORY FLOOR NETWORK — REPORT             ║
╠════════════════════════════════════════════════════════╣
║  Student ID                : 230615005                 ║
║  Dataset seed              : 1244338015                ║
╠════════════════════════════════════════════════════════╣
║  GRAPH STRUCTURE                                       ║
║  Stations (vertices)       : 16                        ║
║  Conveyor paths (edges)    : 54                        ║
║  Graph density             : 0.45                      ║
║  Connected components      : 1                         ║
║  Has cycle                 : YES                       ║
╠════════════════════════════════════════════════════════╣
║  BFS RESULTS                                           ║
║  Source station            : ST01                      ║
║  Targets checked           : 5                         ║
║  Avg hop distance          : 1.8                      

---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about the network or system?

*Your answer here:*Through this script, you have learned how to procedurally generate a network, represent its structure through mathematical matrices, and use fundamental graph algorithms (BFS and DFS) to analyze connectivity and optimize routing.

---

**Q2 — Which graph concept did you find most important, and why?**
Pick one algorithm (BFS, DFS, topological sort, Dijkstra, cycle detection…) and explain what problem it solves.BFS (Breadth-First Search) explored the network level-by-level to find the shortest path between stations, ensuring that the route with the minimum number of "hops" was identified.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?the script generates a completely different unique seed, which in turn creates an entirely new factory network with different station connections, paths, and shortest-route results.

*Your answer here:*
